# 02 — Model training, calibration and evaluation

This notebook loads the prepared historical feature table from Notebook 01.

It compares the class-prior baseline, class-balanced Linear SVM and balanced histogram gradient boosting model, applies expanding temporal calibration to the learned probability candidates, and selects the final probability model by chronological test-set log loss with Brier score as a secondary check.

The selected model and baseline are saved in `models/` for the simulation and retrospective-validation notebooks.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd()
if (current_dir / "data" / "results.csv").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "results.csv").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

outputs_dir = project_root / "outputs"
intermediate_dir = outputs_dir / "intermediate"
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)

model_features_path = (
    intermediate_dir / "model_features.csv"
)
metadata_path = (
    intermediate_dir / "data_pipeline_metadata.json"
)

if not model_features_path.exists() or not metadata_path.exists():
    raise FileNotFoundError(
        "Run 01_data_and_features.ipynb before model training."
    )

df_model = pd.read_csv(
    model_features_path,
    parse_dates=["date"],
)

data_pipeline_metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

evaluation_cutoff = pd.Timestamp(
    data_pipeline_metadata["evaluation_cutoff"]
)
training_rows = int(
    data_pipeline_metadata["training_rows"]
)
test_rows = int(
    data_pipeline_metadata["test_rows"]
)

print(
    "Loaded prepared model rows:",
    len(df_model),
)

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt

features_upgraded = [
    'home_elo',
    'away_elo',
    'home_form_goals_for',
    'home_form_goals_against',
    'away_form_goals_for',
    'away_form_goals_against',
    'match_weight',
    'is_neutral',
]

train_mask_up = df_model['date'] < evaluation_cutoff
test_mask_up = df_model['date'] >= evaluation_cutoff

X_train_up = df_model.loc[train_mask_up, features_upgraded].copy()
X_test_up = df_model.loc[test_mask_up, features_upgraded].copy()
y_train_up = df_model.loc[train_mask_up, 'target'].copy()
y_test_up = df_model.loc[test_mask_up, 'target'].copy()

if len(X_train_up) != training_rows or len(X_test_up) != test_rows:
    raise ValueError('Elo model does not use the expected chronological split.')
if df_model.loc[train_mask_up, 'date'].max() >= df_model.loc[test_mask_up, 'date'].min():
    raise ValueError('Temporal leakage detected after the Elo merge.')

X_upgraded = df_model[features_upgraded]
y_upgraded = df_model['target']

class_labels = np.array([0, 1, 2])
class_names = {
    0: 'Away Win',
    1: 'Draw',
    2: 'Home Win',
}


class BalancedHistGradientBoostingClassifier(ClassifierMixin, BaseEstimator):
    """HistGradientBoostingClassifier with balanced weights computed inside fit.

    Keeping the weighting inside the estimator lets CalibratedClassifierCV train
    the classifier with class balancing while fitting the calibration layer on
    the natural, unweighted calibration distribution.
    """

    def __init__(
        self,
        learning_rate=0.08,
        max_iter=200,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    ):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.max_leaf_nodes = max_leaf_nodes
        self.l2_regularization = l2_regularization
        self.random_state = random_state

    def fit(self, X, y):
        self.model_ = HistGradientBoostingClassifier(
            learning_rate=self.learning_rate,
            max_iter=self.max_iter,
            max_leaf_nodes=self.max_leaf_nodes,
            l2_regularization=self.l2_regularization,
            random_state=self.random_state,
        )

        sample_weights = compute_sample_weight(
            class_weight='balanced',
            y=y,
        )

        self.model_.fit(
            X,
            y,
            sample_weight=sample_weights,
        )
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


# -------------------------------------------------------------------------
# Classification diagnostics
# -------------------------------------------------------------------------

scaler_upgraded = StandardScaler()
X_train_scaled_up = scaler_upgraded.fit_transform(X_train_up)
X_test_scaled_up = scaler_upgraded.transform(X_test_up)

dummy_model = DummyClassifier(
    strategy='prior',
    random_state=42,
)
linear_svm_model = LinearSVC(
    class_weight='balanced',
    random_state=42,
    max_iter=5000,
)
balanced_hgb_model = BalancedHistGradientBoostingClassifier()

print('Training class-prior baseline...')
dummy_model.fit(X_train_up, y_train_up)

print('Training class-balanced Linear SVM...')
linear_svm_model.fit(X_train_scaled_up, y_train_up)

print('Training class-balanced HistGradientBoosting...')
balanced_hgb_model.fit(X_train_up, y_train_up)

classification_models = {
    'Class-prior baseline': (dummy_model, X_test_up),
    'Linear SVM': (linear_svm_model, X_test_scaled_up),
    'Balanced HistGradientBoosting': (balanced_hgb_model, X_test_up),
}

comparison_rows = []
model_predictions = {}
confusion_rows = []
recall_rows = []

for model_name, (model, model_X_test) in classification_models.items():
    predictions = model.predict(model_X_test)
    model_predictions[model_name] = predictions

    comparison_rows.append({
        'model': model_name,
        'accuracy': accuracy_score(y_test_up, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_test_up, predictions),
        'macro_f1': f1_score(y_test_up, predictions, average='macro'),
    })

    recalls = recall_score(
        y_test_up,
        predictions,
        labels=class_labels,
        average=None,
        zero_division=0,
    )

    for label, recall_value in zip(class_labels, recalls):
        recall_rows.append({
            'model': model_name,
            'class_label': int(label),
            'class_name': class_names[int(label)],
            'recall': float(recall_value),
        })

    matrix = confusion_matrix(
        y_test_up,
        predictions,
        labels=class_labels,
    )

    for actual_index, actual_label in enumerate(class_labels):
        for predicted_index, predicted_label in enumerate(class_labels):
            confusion_rows.append({
                'model': model_name,
                'actual_label': int(actual_label),
                'actual_name': class_names[int(actual_label)],
                'predicted_label': int(predicted_label),
                'predicted_name': class_names[int(predicted_label)],
                'count': int(matrix[actual_index, predicted_index]),
            })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        ['macro_f1', 'balanced_accuracy'],
        ascending=False,
    )
    .reset_index(drop=True)
)
model_comparison.to_csv(
    outputs_dir / 'model_comparison_classification.csv',
    index=False,
)

per_class_recall = pd.DataFrame(recall_rows)
per_class_recall.to_csv(
    outputs_dir / 'model_per_class_recall.csv',
    index=False,
)

model_confusion_matrices = pd.DataFrame(confusion_rows)
model_confusion_matrices.to_csv(
    outputs_dir / 'model_confusion_matrices.csv',
    index=False,
)

print('\nClassification comparison')
display(model_comparison)

print('\nDraw recall from hard class predictions')
display(
    per_class_recall.loc[
        per_class_recall['class_label'].eq(1),
        ['model', 'recall'],
    ]
    .sort_values('recall', ascending=False)
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Temporal calibration folds
# -------------------------------------------------------------------------

X_train_probability = X_train_up.reset_index(drop=True)
y_train_probability = y_train_up.reset_index(drop=True)
X_test_probability = X_test_up.reset_index(drop=True)
y_test_probability = y_test_up.reset_index(drop=True)

training_dates_probability = (
    df_model.loc[train_mask_up, 'date']
    .reset_index(drop=True)
)

if not training_dates_probability.is_monotonic_increasing:
    raise ValueError(
        'Training rows are not ordered chronologically; temporal calibration would be invalid.'
    )

unique_training_dates = np.array(
    sorted(training_dates_probability.unique())
)
date_blocks = np.array_split(
    unique_training_dates,
    6,
)

temporal_calibration_folds = []
calibration_fold_rows = []

for fold_number in range(1, 6):
    estimator_dates = np.concatenate(
        date_blocks[:fold_number]
    )
    calibration_dates = date_blocks[fold_number]

    estimator_idx = np.flatnonzero(
        training_dates_probability
        .isin(estimator_dates)
        .to_numpy()
    )
    calibration_idx = np.flatnonzero(
        training_dates_probability
        .isin(calibration_dates)
        .to_numpy()
    )

    estimator_end = pd.Timestamp(estimator_dates[-1])
    calibration_start = pd.Timestamp(calibration_dates[0])

    if estimator_end >= calibration_start:
        raise ValueError(
            f'Calibration fold {fold_number} is not strictly chronological.'
        )

    estimator_classes = set(
        y_train_probability
        .iloc[estimator_idx]
        .astype(int)
        .unique()
    )
    calibration_classes = set(
        y_train_probability
        .iloc[calibration_idx]
        .astype(int)
        .unique()
    )
    expected_classes = set(class_labels.tolist())

    if estimator_classes != expected_classes:
        raise ValueError(
            f'Estimator block {fold_number} is missing an outcome class.'
        )
    if calibration_classes != expected_classes:
        raise ValueError(
            f'Calibration block {fold_number} is missing an outcome class.'
        )

    temporal_calibration_folds.append(
        (estimator_idx, calibration_idx)
    )
    calibration_fold_rows.append({
        'fold': fold_number,
        'estimator_rows': len(estimator_idx),
        'calibration_rows': len(calibration_idx),
        'estimator_start': pd.Timestamp(estimator_dates[0]).strftime('%Y-%m-%d'),
        'estimator_end': estimator_end.strftime('%Y-%m-%d'),
        'calibration_start': calibration_start.strftime('%Y-%m-%d'),
        'calibration_end': pd.Timestamp(calibration_dates[-1]).strftime('%Y-%m-%d'),
        'date_overlap_detected': 0,
    })

calibration_fold_validation = pd.DataFrame(
    calibration_fold_rows
)
calibration_fold_validation.to_csv(
    outputs_dir / 'calibration_fold_validation.csv',
    index=False,
)


# -------------------------------------------------------------------------
# Final calibrated probability candidates
# -------------------------------------------------------------------------

linear_svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(
        class_weight='balanced',
        random_state=42,
        max_iter=5000,
    )),
])

try:
    calibrated_linear_svm_model = CalibratedClassifierCV(
        estimator=linear_svm_pipeline,
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )
    calibrated_balanced_hgb_model = CalibratedClassifierCV(
        estimator=BalancedHistGradientBoostingClassifier(),
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )
except TypeError:
    calibrated_linear_svm_model = CalibratedClassifierCV(
        base_estimator=linear_svm_pipeline,
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )
    calibrated_balanced_hgb_model = CalibratedClassifierCV(
        base_estimator=BalancedHistGradientBoostingClassifier(),
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )

print('Training temporally calibrated Linear SVM...')
calibrated_linear_svm_model.fit(
    X_train_probability,
    y_train_probability,
)

print('Training temporally calibrated balanced HistGradientBoosting...')
calibrated_balanced_hgb_model.fit(
    X_train_probability,
    y_train_probability,
)


def align_probability_columns(model, probabilities, labels):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )
    aligned = np.zeros(
        (len(probabilities), len(labels)),
        dtype=float,
    )

    for source_index, class_label in enumerate(model.classes_):
        destination = np.where(
            labels == int(class_label)
        )[0]

        if len(destination) != 1:
            raise ValueError(
                f'Unexpected model class label: {class_label}'
            )

        aligned[:, destination[0]] = probabilities[:, source_index]

    row_sums = aligned.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        raise ValueError(
            'Predicted probabilities do not sum to one.'
        )

    return aligned


def multiclass_brier_score(
    y_true,
    probabilities,
    labels,
):
    y_array = np.asarray(
        y_true,
        dtype=int,
    )
    one_hot = np.column_stack([
        (y_array == label).astype(float)
        for label in labels
    ])

    return float(
        np.mean(
            np.sum(
                (probabilities - one_hot) ** 2,
                axis=1,
            )
        )
    )


probability_models = {
    'Class-prior baseline': dummy_model,
    'Calibrated Linear SVM': calibrated_linear_svm_model,
    'Calibrated Balanced HistGradientBoosting': calibrated_balanced_hgb_model,
}

probability_comparison_rows = []
probability_predictions = {}
probability_outputs = {}

for model_name, model in probability_models.items():
    raw_probabilities = model.predict_proba(
        X_test_probability
    )
    probabilities = align_probability_columns(
        model,
        raw_probabilities,
        class_labels,
    )
    predictions = class_labels[
        np.argmax(probabilities, axis=1)
    ]

    probability_predictions[model_name] = predictions
    probability_outputs[model_name] = probabilities

    recalls = recall_score(
        y_test_probability,
        predictions,
        labels=class_labels,
        average=None,
        zero_division=0,
    )

    probability_comparison_rows.append({
        'model': model_name,
        'accuracy': accuracy_score(
            y_test_probability,
            predictions,
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_test_probability,
            predictions,
        ),
        'macro_f1': f1_score(
            y_test_probability,
            predictions,
            average='macro',
        ),
        'away_win_recall': float(recalls[0]),
        'draw_recall': float(recalls[1]),
        'home_win_recall': float(recalls[2]),
        'log_loss': log_loss(
            y_test_probability,
            probabilities,
            labels=class_labels,
        ),
        'multiclass_brier': multiclass_brier_score(
            y_test_probability,
            probabilities,
            class_labels,
        ),
    })

probability_model_comparison = (
    pd.DataFrame(probability_comparison_rows)
    .sort_values(
        ['log_loss', 'multiclass_brier'],
        ascending=True,
    )
    .reset_index(drop=True)
)
probability_model_comparison.insert(
    0,
    'probability_rank',
    np.arange(
        1,
        len(probability_model_comparison) + 1,
    ),
)
probability_model_comparison.to_csv(
    outputs_dir / 'probability_model_comparison.csv',
    index=False,
)

selected_probability_model_name = (
    probability_model_comparison.loc[0, 'model']
)
selected_probability_model = probability_models[
    selected_probability_model_name
]
selected_test_probabilities = probability_outputs[
    selected_probability_model_name
]
selected_test_predictions = probability_predictions[
    selected_probability_model_name
]

selected_model_summary = (
    probability_model_comparison
    .iloc[[0]]
    .copy()
)
selected_model_summary.to_csv(
    outputs_dir / 'selected_probability_model.csv',
    index=False,
)

print('\nFinal calibrated probability comparison')
display(probability_model_comparison)
print(
    'Selected probability model:',
    selected_probability_model_name,
)


# -------------------------------------------------------------------------
# Calibration and draw-probability diagnostics
# -------------------------------------------------------------------------

calibration_rows = []

for class_index, class_label in enumerate(class_labels):
    observed_frequency, mean_predicted_probability = calibration_curve(
        (
            np.asarray(
                y_test_probability,
                dtype=int,
            )
            == class_label
        ).astype(int),
        selected_test_probabilities[:, class_index],
        n_bins=10,
        strategy='quantile',
    )

    for bin_number, (predicted, observed) in enumerate(
        zip(
            mean_predicted_probability,
            observed_frequency,
        ),
        start=1,
    ):
        calibration_rows.append({
            'model': selected_probability_model_name,
            'class_label': int(class_label),
            'class_name': class_names[int(class_label)],
            'bin': bin_number,
            'mean_predicted_probability': float(predicted),
            'observed_frequency': float(observed),
        })

selected_model_calibration = pd.DataFrame(
    calibration_rows
)
selected_model_calibration.to_csv(
    outputs_dir / 'selected_model_calibration.csv',
    index=False,
)

draw_curve = selected_model_calibration.loc[
    selected_model_calibration['class_label'].eq(1)
].copy()

selected_draw_recall = recall_score(
    y_test_probability,
    selected_test_predictions,
    labels=class_labels,
    average=None,
    zero_division=0,
)[1]

draw_probability_diagnostic = pd.DataFrame([
    {
        'model': selected_probability_model_name,
        'test_matches': len(y_test_probability),
        'actual_draws': int(
            (np.asarray(y_test_probability) == 1).sum()
        ),
        'actual_draw_rate': float(
            (np.asarray(y_test_probability) == 1).mean()
        ),
        'mean_predicted_draw_probability': float(
            selected_test_probabilities[:, 1].mean()
        ),
        'argmax_draw_prediction_rate': float(
            (selected_test_predictions == 1).mean()
        ),
        'argmax_draw_recall': float(selected_draw_recall),
        'draw_calibration_bin_mae': float(
            np.mean(
                np.abs(
                    draw_curve['mean_predicted_probability']
                    - draw_curve['observed_frequency']
                )
            )
        ),
    }
])
draw_probability_diagnostic.to_csv(
    outputs_dir / 'draw_probability_diagnostic.csv',
    index=False,
)

# Compare the raw balanced HGB probabilities from the classification model with
# the fair, temporally calibrated HGB probabilities.
raw_hgb_probabilities = align_probability_columns(
    balanced_hgb_model,
    balanced_hgb_model.predict_proba(
        X_test_probability
    ),
    class_labels,
)
raw_hgb_predictions = class_labels[
    np.argmax(raw_hgb_probabilities, axis=1)
]
raw_hgb_recalls = recall_score(
    y_test_probability,
    raw_hgb_predictions,
    labels=class_labels,
    average=None,
    zero_division=0,
)

calibrated_hgb_row = (
    probability_model_comparison.loc[
        probability_model_comparison['model'].eq(
            'Calibrated Balanced HistGradientBoosting'
        )
    ]
    .iloc[0]
)

hgb_calibration_sensitivity = pd.DataFrame([
    {
        'variant': 'Balanced HGB raw probabilities',
        'accuracy': accuracy_score(
            y_test_probability,
            raw_hgb_predictions,
        ),
        'macro_f1': f1_score(
            y_test_probability,
            raw_hgb_predictions,
            average='macro',
        ),
        'draw_recall': float(raw_hgb_recalls[1]),
        'log_loss': log_loss(
            y_test_probability,
            raw_hgb_probabilities,
            labels=class_labels,
        ),
        'multiclass_brier': multiclass_brier_score(
            y_test_probability,
            raw_hgb_probabilities,
            class_labels,
        ),
    },
    {
        'variant': 'Balanced HGB temporally calibrated',
        'accuracy': float(calibrated_hgb_row['accuracy']),
        'macro_f1': float(calibrated_hgb_row['macro_f1']),
        'draw_recall': float(calibrated_hgb_row['draw_recall']),
        'log_loss': float(calibrated_hgb_row['log_loss']),
        'multiclass_brier': float(
            calibrated_hgb_row['multiclass_brier']
        ),
    },
])
hgb_calibration_sensitivity.to_csv(
    outputs_dir / 'hgb_calibration_sensitivity.csv',
    index=False,
)

selected_confusion = confusion_matrix(
    y_test_probability,
    selected_test_predictions,
    labels=class_labels,
)
selected_confusion_rows = []

for actual_index, actual_label in enumerate(class_labels):
    for predicted_index, predicted_label in enumerate(class_labels):
        selected_confusion_rows.append({
            'model': selected_probability_model_name,
            'actual_label': int(actual_label),
            'actual_name': class_names[int(actual_label)],
            'predicted_label': int(predicted_label),
            'predicted_name': class_names[int(predicted_label)],
            'count': int(
                selected_confusion[
                    actual_index,
                    predicted_index,
                ]
            ),
        })

pd.DataFrame(
    selected_confusion_rows
).to_csv(
    outputs_dir / 'selected_probability_confusion_matrix.csv',
    index=False,
)

figures_dir = outputs_dir / 'figures'
figures_dir.mkdir(exist_ok=True)

plt.figure(figsize=(7, 6))
plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Perfect calibration',
)

for class_label in class_labels:
    class_curve = selected_model_calibration.loc[
        selected_model_calibration['class_label'].eq(
            int(class_label)
        )
    ]

    plt.plot(
        class_curve['mean_predicted_probability'],
        class_curve['observed_frequency'],
        marker='o',
        label=class_names[int(class_label)],
    )

plt.xlabel('Mean predicted probability')
plt.ylabel('Observed frequency')
plt.title(
    f'Calibration: {selected_probability_model_name}'
)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(
    figures_dir / 'selected_model_calibration.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

plt.figure(figsize=(8, 5))
plot_frame = (
    probability_model_comparison
    .sort_values('log_loss', ascending=False)
)
plt.barh(
    plot_frame['model'],
    plot_frame['log_loss'],
)
plt.xlabel('Multiclass log loss (lower is better)')
plt.ylabel('Model')
plt.title('Final probability model comparison')
plt.tight_layout()
plt.savefig(
    figures_dir / 'final_log_loss_comparison.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

plt.figure(figsize=(8, 5))
draw_plot = (
    per_class_recall.loc[
        per_class_recall['class_label'].eq(1)
    ]
    .sort_values('recall')
)
plt.barh(
    draw_plot['model'],
    draw_plot['recall'] * 100,
)
plt.xlabel('Draw recall from argmax class prediction (%)')
plt.ylabel('Model')
plt.title('Draw detection in classification diagnostics')
plt.tight_layout()
plt.savefig(
    figures_dir / 'classification_draw_recall.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

In [ ]:
import sklearn

selected_model_path = (
    models_dir / "selected_probability_model.joblib"
)
baseline_model_path = (
    models_dir / "class_prior_baseline.joblib"
)

joblib.dump(
    selected_probability_model,
    selected_model_path,
)
joblib.dump(
    dummy_model,
    baseline_model_path,
)

model_metadata = {
    "selected_model_name": selected_probability_model_name,
    "feature_columns": features_upgraded,
    "class_labels": class_labels.tolist(),
    "training_cutoff": evaluation_cutoff.strftime("%Y-%m-%d"),
    "training_end": data_pipeline_metadata["training_end"],
    "holdout_start": data_pipeline_metadata["holdout_start"],
    "holdout_end": data_pipeline_metadata["holdout_end"],
    "state_data_end": data_pipeline_metadata["state_data_end"],
    "training_rows": training_rows,
    "test_rows": test_rows,
    "scikit_learn_version": sklearn.__version__,
}

(models_dir / "model_metadata.json").write_text(
    json.dumps(
        model_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved selected model:",
    selected_model_path,
)
print(
    "Selected model:",
    selected_probability_model_name,
)